# Google Maps Scraper (Playwright + BeautifulSoup)

Instalação:
pip install playwright beautifulsoup4 lxml pandas
playwright install

ATENÇÃO:
- O Google Maps muda frequentemente o HTML interno.
- Este scraper funciona melhor com Playwright controlando um navegador real.
- Scraping em larga escala pode violar os termos do Google.

In [3]:
!apt-get update

!apt-get install -y \
    libnss3 \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libxkbcommon0 \
    libasound2 \
    libpango-1.0-0 \
    libcairo2 \
    libatspi2.0-0 \
    libgtk-3-0

!pip install -q playwright beautifulsoup4 lxml pandas nest_asyncio

!playwright install chromium

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,247 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]


In [14]:
import re
import json
import time
import asyncio
import pandas as pd
import nest_asyncio

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from google.colab import files

nest_asyncio.apply()

In [4]:
DATASET_FILE = "/content/drive/MyDrive/Colab Notebooks/Radar/radar.csv"

In [10]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

CITY = "Fortaleza, CE"     # cidade
CATEGORY = "barbearia"       # categoria
USE_CURRENT_LOCATION = False    # True para usar localização atual
SCROLL_TIMES = 15

In [11]:
# ============================================================
# HELPERS
# ============================================================

def clean_text(text):
    if not text:
        return ""

    return re.sub(r"\s+", " ", text).strip()


def extract_phone(text):

    pattern = r"(\+?\d{1,3}\s?)?(\(?\d{2}\)?\s?)?\d{4,5}[-\s]?\d{4}"

    match = re.search(pattern, text)

    if match:
        return match.group(0)

    return "Não encontrado"


def extract_socials(html):

    soup = BeautifulSoup(html, "lxml")

    socials = {
        "instagram": None,
        "facebook": None,
        "linkedin": None,
        "youtube": None,
        "tiktok": None,
    }

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "instagram.com" in href:
            socials["instagram"] = href

        elif "facebook.com" in href:
            socials["facebook"] = href

        elif "linkedin.com" in href:
            socials["linkedin"] = href

        elif "youtube.com" in href:
            socials["youtube"] = href

        elif "tiktok.com" in href:
            socials["tiktok"] = href

    return socials

def extract_email(text):

    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"

    matches = re.findall(pattern, text)

    blacklist = [
        ".png",
        ".jpg",
        ".jpeg",
        ".webp"
    ]

    for email in matches:

        if not any(x in email.lower() for x in blacklist):
            return email

    return "Não encontrado"

def build_url():

    if USE_CURRENT_LOCATION:
        query = f"{CATEGORY} perto de mim"
    else:
        query = f"{CATEGORY} em {CITY}"

    return f"https://www.google.com/maps/search/{query.replace(' ', '+')}"

In [12]:
# ============================================================
# SCRAPER
# ============================================================

async def scrape():

    url = build_url()

    # ========================================================
    # CARREGAR DATASET EXISTENTE
    # ========================================================

    try:

        existing_df = pd.read_csv(DATASET_FILE)

        existing_urls = set(
            existing_df["maps_url"].astype(str).tolist()
        )

        results = existing_df.to_dict("records")

        print("\nDataset existente carregado.")
        print(f"{len(results)} registros já existentes.\n")

    except:

        existing_df = pd.DataFrame()

        existing_urls = set()

        results = []

        print("\nNenhum dataset encontrado.")
        print("Criando novo dataset.\n")

    # ========================================================
    # ABRIR GOOGLE MAPS
    # ========================================================

    print(f"\nAbrindo Google Maps:\n{url}\n")

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu"
            ]
        )

        context = await browser.new_context(
            locale="pt-BR"
        )

        page = await context.new_page()

        await page.goto(url, timeout=120000)

        await page.wait_for_timeout(6000)

        # ====================================================
        # SCROLL NA LISTA
        # ====================================================

        try:

            scrollable = page.locator('div[role="feed"]').first

            for scroll in range(SCROLL_TIMES):

                print(f"Scroll {scroll+1}/{SCROLL_TIMES}")

                await scrollable.evaluate(
                    "(el) => el.scrollBy(0, el.scrollHeight)"
                )

                await page.wait_for_timeout(2000)

        except Exception as e:

            print("\nErro no scroll:")
            print(str(e))

        # ====================================================
        # PEGAR CARDS
        # ====================================================

        cards = page.locator('a[href*="/place/"]')

        total = await cards.count()

        print(f"\n{total} locais encontrados.\n")

        visited = set()

        # ====================================================
        # LOOP PRINCIPAL
        # ====================================================

        for i in range(total):

            try:

                print("\n")
                print("#"*80)
                print(f"PROCESSANDO ITEM {i+1} DE {total}")
                print("#"*80)

                card = cards.nth(i)

                href = await card.get_attribute("href")

                if not href:
                    print("Sem href.")
                    continue

                if href in visited:
                    print("Item repetido.")
                    continue

                visited.add(href)

                await card.click()

                await page.wait_for_timeout(4000)

                html = await page.content()

                soup = BeautifulSoup(html, "lxml")

                body_text = clean_text(
                    soup.get_text(" ")
                )

                # ============================================
                # NOME
                # ============================================

                name = "Não encontrado"

                h1 = soup.find("h1")

                if h1:
                    name = clean_text(h1.text)

                # ============================================
                # TELEFONE
                # ============================================

                phone = extract_phone(body_text)

                # ============================================
                # EMAIL
                # ============================================

                email = extract_email(body_text)

                # ============================================
                # WEBSITE
                # ============================================

                website = "Não possui"

                for a in soup.find_all("a", href=True):

                    href_link = a["href"]

                    if (
                        href_link.startswith("http")
                        and "google" not in href_link
                    ):
                        website = href_link
                        break

                # ============================================
                # REDES SOCIAIS
                # ============================================

                socials = extract_socials(html)

                # ============================================
                # ENDEREÇO
                # ============================================

                address = "Não encontrado"

                address_match = re.search(
                    r"(Rua|R\.|Avenida|Av\.).*?\d+",
                    body_text
                )

                if address_match:
                    address = address_match.group(0)

                # ============================================
                # DATA
                # ============================================

                data = {
                    "categoria": CATEGORY,
                    "cidade": CITY if not USE_CURRENT_LOCATION else "Localização Atual",
                    "nome": name,
                    "telefone": phone,
                    "email": email,
                    "website": website,
                    "endereco": address,
                    "instagram": socials["instagram"],
                    "facebook": socials["facebook"],
                    "linkedin": socials["linkedin"],
                    "youtube": socials["youtube"],
                    "tiktok": socials["tiktok"],
                    "maps_url": page.url
                }

                # ============================================
                # EVITAR DUPLICADOS
                # ============================================

                if data["maps_url"] not in existing_urls:

                    results.append(data)

                    existing_urls.add(data["maps_url"])

                    # ========================================
                    # SAVE INCREMENTAL
                    # ========================================

                    temp_df = pd.DataFrame(results)

                    temp_df.to_csv(
                        DATASET_FILE,
                        index=False
                    )

                    # ========================================
                    # LOG
                    # ========================================

                    print("\n" + "="*70)
                    print(f"ITEM {len(results)} ADICIONADO")
                    print("="*70)

                    print(f"Categoria : {data['categoria']}")
                    print(f"Cidade    : {data['cidade']}")
                    print(f"Nome      : {data['nome']}")
                    print(f"Telefone  : {data['telefone']}")
                    print(f"Email  : {data['email']}")
                    print(f"Website   : {data['website']}")
                    print(f"Endereço  : {data['endereco']}")

                    print("\nREDES SOCIAIS")

                    print(f"Instagram : {data['instagram']}")
                    print(f"Facebook  : {data['facebook']}")
                    print(f"LinkedIn  : {data['linkedin']}")
                    print(f"YouTube   : {data['youtube']}")
                    print(f"TikTok    : {data['tiktok']}")

                    print(f"\nTOTAL DATASET: {len(results)}")

                    print("="*70)

                else:

                    print("\n[SKIP] Loja já existe no dataset.")

            except Exception as e:

                print("\n")
                print("!"*80)
                print(f"ERRO AO PROCESSAR ITEM {i+1}")
                print(str(e))
                print("!"*80)

        await browser.close()

    return results

In [15]:

# ============================================================
# EXECUTAR
# ============================================================

results = await scrape()


Nenhum dataset encontrado.
Criando novo dataset.


Abrindo Google Maps:
https://www.google.com/maps/search/barbearia+em+Fortaleza,+CE

Scroll 1/15
Scroll 2/15
Scroll 3/15
Scroll 4/15
Scroll 5/15
Scroll 6/15
Scroll 7/15
Scroll 8/15
Scroll 9/15
Scroll 10/15
Scroll 11/15
Scroll 12/15
Scroll 13/15
Scroll 14/15
Scroll 15/15

72 locais encontrados.



################################################################################
PROCESSANDO ITEM 1 DE 72
################################################################################

ITEM 1 ADICIONADO
Categoria : barbearia
Cidade    : Fortaleza, CE
Nome      : Resultados
Telefone  : (85) 99917-7002
Email  : Não encontrado
Website   : https://booksy.com/pt-br/rwg/137845_barber-shop-old-cut_barbearias_278919_fortaleza?rwg_token=AFd1xnHnMRzfT9Xtfo1N-aji1wWjtxywRzE9aRIJk9wz6Czw-0oP5mSI94yHxKuFU9bljf-PUakPVXqTI2xsQlhVFiQvIA6ZPg%3D%3D
Endereço  : Av. Dom Luís, 500

REDES SOCIAIS
Instagram : https://www.instagram.com/ivanbarbershopofc?igsh=cnE0a

In [16]:
# ============================================================
# DATAFRAME FINAL
# ============================================================

df = pd.DataFrame(results)

print("\n")
print("="*80)
print("SCRAPING FINALIZADO")
print("="*80)

print(f"\nTOTAL FINAL DE REGISTROS: {len(df)}")

display(df.head())

# ============================================================
# EXPORTAR JSON
# ============================================================

json_file = "google_maps_master.json"

with open(json_file, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# DOWNLOAD
# ============================================================

files.download(DATASET_FILE)
files.download(json_file)

print("\nArquivos exportados com sucesso.")



SCRAPING FINALIZADO

TOTAL FINAL DE REGISTROS: 47


,categoria,cidade,nome,telefone,email,website,endereco,instagram,facebook,linkedin,youtube,tiktok,maps_url
0,barbearia,"Fortaleza, CE",Resultados,(85) 99917-7002,Não encontrado,https://booksy.com/pt-br/rwg/137845_barber-sho...,"Av. Dom Luís, 500",https://www.instagram.com/ivanbarbershopofc?ig...,None,None,None,None,https://www.google.com/maps/place/Ivan+Barbers...
1,barbearia,"Fortaleza, CE",Resultados,(85) 99917-7002,Não encontrado,https://booksy.com/pt-br/rwg/137845_barber-sho...,"Av. Dom Luís, 500",None,None,None,None,None,https://www.google.com/maps/place/Z%C3%A9+Barb...
2,barbearia,"Fortaleza, CE",Resultados,(85) 99917-7002,Não encontrado,https://booksy.com/pt-br/rwg/137845_barber-sho...,"Av. Dom Luís, 500",None,None,None,None,None,https://www.google.com/maps/place/Sullivan+Bar...
3,barbearia,"Fortaleza, CE",Resultados,(85) 99917-7002,Não encontrado,https://booksy.com/pt-br/rwg/137845_barber-sho...,"Av. Dom Luís, 500",None,None,None,None,None,https://www.google.com/maps/place/Baker+Street...
4,barbearia,"Fortaleza, CE",Resultados,(85) 99917-7002,Não encontrado,https://booksy.com/pt-br/rwg/137845_barber-sho...,"Av. Dom Luís, 500",None,None,None,None,None,https://www.google.com/maps/place/Barber+Shop+...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Arquivos exportados com sucesso.
